# 03 - Does writing tables value by value help retrieval?

Same filings, same questions, same models. Only one thing changes: how tables are written in the chunks
* `text/` = cells flattened into a run of numbers;
* `tables/` = every value with its row and column (version 1);
* `tables_v2/` = version 1 plus fixes: better titles, units, split tables, aligned percentage rows, small tables
  merged into the paragraph before them.

The question is only about **finding the right passages** (no language model, no answer grading here).

How the comparison is made:
* the gold evidence of each question points to chunks of the text version; for the table versions it is
  translated by matching content (`src/eval/remap.py`). It is an approximation, so the questions where the
  match was weak are listed and the result is also shown without them;
* because that gold is approximate, a second measure does not use it: **numbers in top 5** = the share of questions
  where every number of the written gold answer (3 or more digits, not years) appears in the 5 best passages;
* three retrieval settings, from simple to complete: dense only; filing filter + reranker; filing filter +
  keyword search + reranker;
* the reranker limit is 1024 tokens for all versions, so no table chunk is cut.

**Before running:** Runtime > T4 GPU; Colab Secret `HF_TOKEN`. Needs notebooks `02a`, `02b` and `02c` to be done.


In [ ]:
import os
import subprocess
import sys

from google.colab import userdata

REPO = "10K-agentic-rag"
BRANCH = "main"
try:  # only needed while the repository is private (a read-only token for this one repository)
    token = userdata.get("GITHUB_TOKEN")
except Exception:  # noqa: BLE001  the secret does not exist: the repository is public
    token = None
login = f"{token}@" if token else ""
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://{login}github.com/ilaria-cesa/{REPO}.git"],
        check=True,
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(torch.cuda.get_device_name(0))

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never hard-code the token
HF_REPO = "IlariaCesa/10k-agentic-rag-data"

Load the three versions of the chunks and vectors from the dataset, and the 175 questions from the repository.

In [ ]:
from src.eval.golden import load_golden, validate
from src.eval.remap import remap_questions
from src.hub.dataset import download_folder
from src.retrieval.embed import load_chunks

for folder in ["text", "tables", "tables_v2"]:
    download_folder(HF_REPO, folder, Path("data"))
VARIANTS = {"text": "data/text", "tables": "data/tables", "tables_v2": "data/tables_v2"}
CHUNK_FILES = {
    "text": "chunks_1600.jsonl",
    "tables": "chunks_tables_1600.jsonl",
    "tables_v2": "chunks_tables_v2_1600.jsonl",
}
chunks = {v: load_chunks(f"{p}/{CHUNK_FILES[v]}") for v, p in VARIANTS.items()}
vectors = {v: np.load(f"{p}/embeddings_bge-m3.npy").astype(np.float32) for v, p in VARIANTS.items()}
for v in VARIANTS:
    assert len(chunks[v]) == len(vectors[v]), v
    print(v, vectors[v].shape)

all_questions = load_golden(Path("questions/questions.jsonl"))
errors, _ = validate(all_questions, chunks["text"])
assert not errors, errors
answerable = [q for q in all_questions if q["expected"] == "answer"]
gold = {"text": answerable}
weak, gold_whole = {}, {}
# Main gold: for each question, the new chunks that contain the lines of the old evidence which hold the
# numbers of the written answer (0.7 of the line's content). Second gold (whole old chunk, 0.5): coarser, kept
# to show how much the result depends on how the evidence is translated.
for v in ["tables", "tables_v2"]:
    gold[v], weak[v] = remap_questions(
        answerable, chunks["text"], chunks[v], min_cover=0.7, focus_on_answer=True
    )
    gold_whole[v], _ = remap_questions(answerable, chunks["text"], chunks[v])
    print(
        v,
        "- weak matches:",
        len(weak[v]),
        "- gold chunks per question:",
        round(np.mean([len(q["evidence_chunk_ids"]) for q in gold[v]]), 2),
    )
print(
    len(answerable),
    "answerable questions; text gold chunks per question:",
    round(np.mean([len(q["evidence_chunk_ids"]) for q in gold["text"]]), 2),
)

Models: the same bge-m3 embeds the questions, and the same reranker reorders candidates for both versions.

In [ ]:
from sentence_transformers import SentenceTransformer

from src.retrieval.embed import embed_texts
from src.retrieval.pipeline import Corpus, RetrievalConfig, retrieve
from src.retrieval.rerank import Reranker

model = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
model.max_seq_length = 1024
query_vectors = embed_texts(model, [q["question"] for q in answerable], batch_size=32)
reranker = Reranker(max_length=1024)
corpora = {v: Corpus(chunks[v], vectors[v]) for v in VARIANTS}

Run the three settings on the three versions. For every question we keep the ranked chunk ids.

In [ ]:
import re

from src.eval.metrics import retrieval_metrics

CONFIGS = [
    RetrievalConfig("dense only"),
    RetrievalConfig("filter + rerank", use_filter=True, rerank=True),
    RetrievalConfig("filter + hybrid + rerank", use_filter=True, hybrid=True, rerank=True),
]
NUMBER = re.compile(r"\d[\d,]*\.?\d*")


def numbers(text):
    """Numbers of 3+ digits (decimal point counts), without years, commas removed."""
    found = {n.rstrip(".,").replace(",", "") for n in NUMBER.findall(text)}
    return {n for n in found if len(n) >= 3 and not re.fullmatch(r"(19|20)\d\d", n)}


chunk_text = {v: {c["chunk_id"]: c["text"] for c in chunks[v]} for v in VARIANTS}
rows = []
for variant in VARIANTS:
    for config in CONFIGS:
        for q, vec in zip(gold[variant], query_vectors):
            idx, _ = retrieve(corpora[variant], q["question"], vec, config, reranker)
            ids = [chunks[variant][i]["chunk_id"] for i in idx]
            wanted = numbers(q["answer"])
            shown = set().union(*(numbers(chunk_text[variant][i]) for i in ids[:5]))
            rows.append(
                {
                    "variant": variant,
                    "config": config.name,
                    "id": q["id"],
                    "split": q["split"],
                    "type": q["type"],
                    "weak": q["id"] in weak.get(variant, ()),
                    # independent of the remapped gold; undefined when the answer has no number
                    "numbers_top5": float(wanted <= shown) if wanted else np.nan,
                    "ranked": ids,
                }
                | retrieval_metrics(q["evidence_chunk_ids"], ids)
            )
df = pd.DataFrame(rows)
cols = ["hit@1", "hit@5", "hit@10", "recall@5", "all@5", "mrr", "numbers_top5"]
print(df.groupby(["config", "variant"])[cols].mean().round(3).to_string())
# Same rankings, scored against the coarser gold (table versions only).
for v in ["tables", "tables_v2"]:
    whole = {q["id"]: q["evidence_chunk_ids"] for q in gold_whole[v]}
    sub = df[df["variant"] == v]
    extra = pd.DataFrame([retrieval_metrics(whole[i], r) for i, r in zip(sub["id"], sub["ranked"])])
    extra["config"] = sub["config"].to_numpy()
    print(f"\n{v} scored against the coarser gold (whole old chunk):")
    print(extra.groupby("config")[cols[:-1]].mean().round(3).to_string())

The same, without the questions whose table match was weak (in either table version), and by question type.

In [ ]:
weak_ids = set().union(*weak.values())
strong = df[~df["id"].isin(weak_ids)]
print("Without weak matches:", strong["id"].nunique(), "questions")
print(strong.groupby(["config", "variant"])[cols].mean().round(3).to_string())
sub = df[df["config"] == "filter + hybrid + rerank"]
for metric in ["hit@5", "mrr", "numbers_top5"]:
    print("\n" + metric + " by question type (filter + hybrid + rerank)")
    print(sub.groupby(["type", "variant"])[metric].mean().unstack().round(3).to_string())
print("\nhit@5 by split (filter + hybrid + rerank)")
print(sub.groupby(["split", "variant"])["hit@5"].mean().unstack().round(3).to_string())

Paired view: for each question, is one version better than the other (hit@5 and numbers in top 5)? Same
questions, so this is a fairer test than comparing two averages. The main line to read is `tables_v2` against
`text`; `tables_v2` against `tables` shows whether the fixes helped.


In [ ]:
from math import comb


def sign_test(better, worse):
    n = better + worse
    if n == 0:
        return 1.0
    k = min(better, worse)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2**n)


PAIRS = [("tables", "text"), ("tables_v2", "text"), ("tables_v2", "tables")]
for metric in ["hit@5", "numbers_top5"]:
    print(metric)
    for config in [c.name for c in CONFIGS]:
        wide = (
            df[df["config"] == config].pivot(index="id", columns="variant", values=metric).dropna()
        )
        for a, b in PAIRS:
            better = int(((wide[a] == 1) & (wide[b] == 0)).sum())
            worse = int(((wide[a] == 0) & (wide[b] == 1)).sum())
            print(
                f"  {config:26s} {a:9s} better on {better:2d}, {b:9s} better on {worse:2d}, "
                f"sign test p={sign_test(better, worse):.3f}"
            )

Save the per-question ranked ids to look at the questions that still fail.

In [ ]:
os.makedirs("results", exist_ok=True)
df.to_json("results/retrieval_text_tables_v2.jsonl", orient="records", lines=True)
import shutil

from google.colab import files

shutil.make_archive("retrieval_text_tables_v2", "zip", "results")
files.download("retrieval_text_tables_v2.zip")

Copy back to the chat: the printed tables. The zip has the ranked chunk ids of every question and setting.